# 56 — Preference Learning with DPO

## Goal
Train a model to prefer **better** responses over worse ones using
**Direct Preference Optimization (DPO)** — no reward model needed.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **DPO** | Align model to human preferences directly |
| **Preference pairs** | (prompt, chosen, rejected) format |
| **DPOTrainer** | TRL's high-level DPO implementation |
| **SFT → DPO pipeline** | First SFT, then refine with DPO |

## How DPO Works

Traditional RLHF: Train reward model → Use PPO to optimize policy → Complex

DPO shortcut: Directly optimize the model using preference pairs:
- **Chosen**: The response humans prefer
- **Rejected**: The response humans dislike
- The model learns to increase P(chosen) and decrease P(rejected)

## Stack
- `transformers` + `peft` + `trl`
- Model: **Qwen2.5-0.5B-Instruct**

In [ ]:
import os, warnings
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

import torch
print(f"PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}")

## Step 1 — Create Preference Dataset

Each example has:
- **prompt**: the user's question
- **chosen**: the response we prefer (concise, helpful, professional)
- **rejected**: the response we don't want (verbose, unhelpful, or rude)

Theme: **Customer support tone** — we prefer empathetic, concise, action-oriented.

In [ ]:
preference_data = [
    {"prompt": "I was charged twice for my subscription.",
     "chosen": "I'm sorry about the duplicate charge. I've initiated a refund that will appear in 3-5 business days. Is there anything else I can help with?",
     "rejected": "So you're saying you were charged twice? That's unusual. I'll need to look into this. Can you provide your account number, the date of the charges, the amount, and a screenshot of your bank statement? Also, which payment method did you use? We have a process for this that typically takes 7-14 business days."},

    {"prompt": "Your app keeps crashing on my phone.",
     "chosen": "Sorry about the crashes! Try clearing the app cache (Settings → App → Clear Cache) and updating to the latest version. If it persists, let me know your phone model and OS version so I can investigate.",
     "rejected": "Have you tried turning your phone off and on again? Also check if you have enough storage. Make sure your OS is updated. Clear the cache. Reinstall the app. Check your internet connection. If none of that works, it's probably a device compatibility issue and there's not much we can do."},

    {"prompt": "I want to cancel my account.",
     "chosen": "I understand — you can cancel anytime from Settings → Billing → Cancel Plan. Your access continues until the end of your billing cycle. If there's anything we could improve, I'd love to hear your feedback.",
     "rejected": "Are you sure? You'll lose access to all your data. Maybe you should consider downgrading instead? We have a lot of great features coming soon that I think you'd really like. Can I schedule a call to discuss? I really don't think cancelling is the right move here."},

    {"prompt": "The report I exported has missing data.",
     "chosen": "That's concerning. Can you share the export settings (date range, filters)? I'll run a comparison against our database and get back to you within 4 hours with either the corrected export or an explanation.",
     "rejected": "Reports can sometimes have issues. Did you check that your filters were set correctly? A lot of times users accidentally filter out data. Also, make sure you selected the right date range. If you still have issues, you can try exporting again."},

    {"prompt": "How do I add a team member?",
     "chosen": "Go to Settings → Team → Invite Member. Enter their email, choose a role (Admin, Editor, or Viewer), and they'll receive an invite link valid for 7 days.",
     "rejected": "Sure, so first you'll want to navigate to your dashboard, then click on the gear icon, that takes you to settings, then look for the section that says Team Management, it should be on the left sidebar, scroll down if you don't see it, then click Invite, a form will pop up, fill in the email..."},

    {"prompt": "This is the third time I'm writing about the same issue!",
     "chosen": "You're absolutely right to be frustrated, and I sincerely apologize. Having to write three times is unacceptable. I've escalated this with a 24-hour SLA and will personally follow up tomorrow with a status update.",
     "rejected": "I see you've contacted us about this before. Let me pull up the previous tickets. It looks like the issue was marked as resolved. Can you describe the problem again so I can understand what's happening?"},

    {"prompt": "Is my data secure with your service?",
     "chosen": "Absolutely. We use AES-256 encryption at rest, TLS 1.3 in transit, and are SOC 2 Type II certified. We do daily backups with 30-day retention. Full details at security.example.com.",
     "rejected": "Yes, we take security very seriously. We use industry-standard practices to protect your data. Rest assured that we follow best practices and have security measures in place. You can trust us with your data."},

    {"prompt": "I accidentally deleted my project.",
     "chosen": "Don't worry — we keep backups for 30 days. I've submitted a recovery request and your project should be restored within 2 hours. I'll confirm once it's back.",
     "rejected": "Unfortunately, when you delete a project, it's gone. We don't currently have a way to recover deleted projects. In the future, I'd recommend being more careful when deleting things. You might want to set up your own backups."},

    {"prompt": "Your API keeps returning 500 errors.",
     "chosen": "You're right — we identified an issue with our API gateway this morning. The team deployed a fix and we're monitoring. If you're still seeing errors, share the request IDs and I'll trace them directly.",
     "rejected": "500 errors can be caused by many things. Are you sure it's on our end? Can you check your request format? Make sure your API key is valid. Try again in a few minutes. If it persists, open a ticket."},

    {"prompt": "Can I get an invoice for tax purposes?",
     "chosen": "Of course! Download invoices from Settings → Billing → Invoice History. Need a custom format with your company VAT number? Send me the details and I'll generate one within 24 hours.",
     "rejected": "Invoices can be found somewhere in the settings. I think it's under billing but I'm not 100% sure. You might need to contact our billing department directly. Their email is billing@example.com."},

    {"prompt": "I've been waiting 3 days for a response.",
     "chosen": "You're right — a 3-day wait is not okay, and I'm sorry. I'm prioritizing your ticket right now. What's the issue you need help with? I'll stay on this until it's resolved.",
     "rejected": "We've been experiencing higher than usual ticket volume. Thank you for your patience. Your ticket is in the queue and will be handled in the order it was received. We appreciate your understanding."},

    {"prompt": "Do you have a mobile app?",
     "chosen": "Not yet, but it's coming! Our iOS app is in beta with Android following in Q3. I've added you to the beta waitlist. For now, our web app is fully responsive on mobile browsers.",
     "rejected": "We don't have a mobile app at this time. We're considering developing one in the future, but I can't provide a timeline. You can access our service through a mobile browser for now, though the experience might not be optimal."},
]

print(f"Created {len(preference_data)} preference pairs")

In [ ]:
from datasets import Dataset

# DPO expects: prompt (list of messages), chosen (list), rejected (list)
dpo_formatted = []
for ex in preference_data:
    dpo_formatted.append({
        "prompt": [{"role": "user", "content": ex["prompt"]}],
        "chosen": [{"role": "assistant", "content": ex["chosen"]}],
        "rejected": [{"role": "assistant", "content": ex["rejected"]}],
    })

dpo_dataset = Dataset.from_list(dpo_formatted)
print(f"DPO dataset: {dpo_dataset}")
print(f"\nExample:")
print(f"  Prompt:   {dpo_dataset[0]['prompt'][0]['content'][:60]}...")
print(f"  Chosen:   {dpo_dataset[0]['chosen'][0]['content'][:60]}...")
print(f"  Rejected: {dpo_dataset[0]['rejected'][0]['content'][:60]}...")

## Step 2 — Train with DPO

Key DPO parameters:
- **beta**: Controls deviation from reference model (lower = more freedom)
- **loss_type**: `sigmoid` is default DPO loss
- No reward model needed!

In [ ]:
from peft import LoraConfig
from trl import DPOTrainer, DPOConfig

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "./dpo_preference_model"

dpo_config = DPOConfig(
    output_dir=OUTPUT_DIR,
    max_steps=60,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,              # lower LR for DPO
    lr_scheduler_type="cosine",
    warmup_steps=5,
    beta=0.1,                         # KL penalty strength
    loss_type=["sigmoid"],            # standard DPO loss
    max_length=512,
    bf16=True,
    logging_steps=5,
    gradient_checkpointing=True,
    report_to="none",
    seed=42,
)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

trainer = DPOTrainer(
    model=MODEL_ID,
    args=dpo_config,
    train_dataset=dpo_dataset,
    peft_config=lora_config,
)

print("Starting DPO training...")
print(f"Beta: {dpo_config.beta} (KL penalty)")
trainer.train()
trainer.save_model(OUTPUT_DIR)
print("\n✅ DPO training complete!")

## Step 3 — Compare Base vs DPO Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import AutoPeftModelForCausalLM

# Load DPO model
dpo_model = AutoPeftModelForCausalLM.from_pretrained(OUTPUT_DIR, torch_dtype=torch.bfloat16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Load base model for comparison
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")

def generate(model, prompt):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

# Compare on new prompts
test_prompts = [
    "My payment failed but my card was charged.",
    "Your product is too expensive compared to competitors.",
    "I can't figure out how to use the dashboard.",
]

for prompt in test_prompts:
    print(f"\n{'='*70}")
    print(f"Customer: {prompt}")
    print(f"{'='*70}")
    print(f"\n🔵 BASE MODEL:")
    print(generate(base_model, prompt))
    print(f"\n🟢 DPO MODEL:")
    print(generate(dpo_model, prompt))

## Step 4 — Training Metrics Analysis

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history

# Extract DPO-specific metrics
loss_logs = [l for l in logs if "loss" in l]
reward_logs = [l for l in logs if "rewards/margins" in l]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
if loss_logs:
    axes[0].plot([l["step"] for l in loss_logs], [l["loss"] for l in loss_logs], "b-o")
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("DPO Loss")
    axes[0].set_title("Training Loss")
    axes[0].grid(True, alpha=0.3)

# Reward margins (chosen - rejected)
if reward_logs:
    axes[1].plot([l["step"] for l in reward_logs], [l["rewards/margins"] for l in reward_logs], "g-o")
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Reward Margin")
    axes[1].set_title("Chosen - Rejected Reward Margin")
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print key metrics
if reward_logs:
    acc_logs = [l for l in logs if "rewards/accuracies" in l]
    if acc_logs:
        print(f"Final reward accuracy: {acc_logs[-1]['rewards/accuracies']:.2%}")
    print(f"Final reward margin:  {reward_logs[-1]['rewards/margins']:.4f}")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **DPO** | Direct preference optimization — no reward model needed |
| **Preference pairs** | (prompt, chosen, rejected) → model learns which is better |
| **Beta parameter** | Controls KL divergence from reference model |
| **Reward margins** | Should increase during training (chosen > rejected) |

## DPO vs RLHF vs SFT
| Method | What it does | Complexity |
|--------|-------------|------------|
| **SFT** | Learn to generate target outputs | Low |
| **DPO** | Learn preferences between outputs | Medium |
| **RLHF** | Train reward model → PPO optimization | High |

## When to use DPO
- You have preference data (A vs B rankings)
- You want to refine **style/tone** without changing factuality
- RLHF is too complex for your use case
- You already have an SFT model and want alignment